# Multi-Category Benchmark — MVTec AD

Evaluates the PatchCore pipeline across all 15 MVTec AD categories.

## Goals
- Run PatchCore (multiscale ResNet18) on all 15 categories
- Compare AUROC across scoring strategies: `mean`, `max`, `topk_mean`
- Identify easy vs hard categories
- Visualize anomaly heatmaps on representative categories
- Save results (CSV + JSON) for the final comparison report

## Usage
- **Local (VSCode)**: set `ENV = 'local'` and `LOCAL_DATA_DIR` below
- **Colab**: set `ENV = 'colab'`, run the Colab Setup cell first

## 0. Colab Setup
Run this cell only when executing on Google Colab.
Skip it entirely in local VSCode.

In [ ]:
# COLAB SETUP — run this cell first, only on Google Colab
# Skip entirely if running locally in VSCode

import os, sys

IN_COLAB = "google.colab" in str(get_ipython())

if IN_COLAB:
    # 1. Mount Google Drive
    from google.colab import drive

    drive.mount("/content/drive", force_remount=True)

    # 2. Clone private repo
    # WARNING: never commit this token — paste it here only at runtime
    GITHUB_TOKEN = "ghp_XXXXXXXXXXXXX"  # <-- paste your token here
    REPO = "Machoudi-1/industrial-anomaly-detection"
    REPO_DIR = "/content/industrial-anomaly-detection"

    if not os.path.exists(REPO_DIR):
        ret = os.system(
            f"git clone https://{GITHUB_TOKEN}@github.com/{REPO}.git {REPO_DIR}"
        )
        print("Repo cloned." if ret == 0 else "ERROR: clone failed.")
    else:
        os.system(f"cd {REPO_DIR} && git pull")
        print("Repo updated.")

    # 3. sys.path — point to repo root
    if REPO_DIR not in sys.path:
        sys.path.insert(0, REPO_DIR)
    os.chdir(f"{REPO_DIR}/notebooks")
    print("Working directory:", os.getcwd())
    print("sys.path[0]:", sys.path[0])

    # 4. Install dependencies
    os.system("pip install -q tqdm scikit-learn torchvision pandas matplotlib")
    print("Dependencies OK.")

    # 5. Dataset — extract from Drive zip
    # Zip location on your Drive (adapt if path differs)
    ZIP_PATH = "/content/drive/MyDrive/anomaly_detection_project/datasets/mvtec_ad.zip"
    COLAB_DATA_DIR = "/content/mvtec_ad"

    if not os.path.exists(COLAB_DATA_DIR):
        print("Extracting dataset (~5 GB)...")
        os.makedirs(COLAB_DATA_DIR, exist_ok=True)
        os.system(f"unzip -q {ZIP_PATH} -d {COLAB_DATA_DIR}")
        print("Done.")
    else:
        print("Dataset already extracted.")

    # The zip creates a subfolder mvtec_ad/mvtec_ad — set the real data path
    COLAB_DATA_DIR = "/content/mvtec_ad/mvtec_ad"
    categories_found = [
        c for c in sorted(os.listdir(COLAB_DATA_DIR)) if not c.endswith(".txt")
    ]
    print(f"Categories found ({len(categories_found)}):", categories_found)

    # 6. Output dir on Drive
    COLAB_OUTPUT_DIR = "/content/drive/MyDrive/anomaly_outputs"
    os.makedirs(COLAB_OUTPUT_DIR, exist_ok=True)
    print("Outputs:", COLAB_OUTPUT_DIR)

else:
    print("Running locally — Colab setup skipped.")

## 1. Imports

In [ ]:
import sys
import os

# Colab: REPO_DIR already added in Setup cell above
# Local: go up one level from notebooks/
repo_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

print("repo_root:", repo_root)

In [ ]:
import json
from datetime import datetime
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import torch
from sklearn.metrics import roc_auc_score
from torch.utils.data import DataLoader
from tqdm import tqdm

from mvtec_dataset.mvtec import MvtecAdDataset
from evaluation.metrics import compute_patchcore_scores
from models.patchcore import (
    FeatureExtractor,
    MultiScaleFeatureExtractor,
    build_memory_bank,
    random_coreset_sampling,
)
from utils.normalization import preprocess_image

print("All imports OK")

## 2. Device

In [ ]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

if DEVICE.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## 3. Configuration

Set `ENV = 'local'` or `ENV = 'colab'` before running.

In [ ]:
# Environment
# Switch between 'local' and 'colab' — only thing to change
ENV = "colab"  # 'local' or 'colab'

# Local paths
LOCAL_DATA_DIR = "../data/mvtec_ad"
LOCAL_OUTPUT_DIR = "../outputs"

ENV = "colab"
COLAB_DATA_DIR = "/content/mvtec_ad/mvtec_ad"
COLAB_OUTPUT_DIR = "/content/drive/MyDrive/anomaly_outputs"

# Resolve paths
if ENV == "colab":
    DATA_DIR = COLAB_DATA_DIR
    OUTPUT_DIR = Path(COLAB_OUTPUT_DIR)
else:
    DATA_DIR = LOCAL_DATA_DIR
    OUTPUT_DIR = Path(LOCAL_OUTPUT_DIR)

# All 15 MVTec categories
CATEGORIES = [
    "bottle",
    "cable",
    "capsule",
    "carpet",
    "grid",
    "hazelnut",
    "leather",
    "metal_nut",
    "pill",
    "screw",
    "tile",
    "toothbrush",
    "transistor",
    "wood",
    "zipper",
]

# PatchCore settings
BATCH_SIZE = 32
EXTRACTOR_TYPE = "multiscale"  # 'layer3' or 'multiscale'
SCORING_METHODS = ["mean", "max", "topk_mean"]
K_RATIO = 0.01

USE_RANDOM_CORESET = False
CORESET_RATIO = 0.1

# Output options
SAVE_RESULTS = True
SAVE_MEMORY_BANKS = True

RUN_NAME = f"{datetime.now().strftime('%Y%m%d_%H%M%S')}_multicategory_patchcore"
print("Run:", RUN_NAME)
print("DATA_DIR:", DATA_DIR)
print("OUTPUT_DIR:", OUTPUT_DIR)

## 4. Output Folders

In [ ]:
METRICS_DIR = OUTPUT_DIR / "metrics"
MEMORY_DIR = OUTPUT_DIR / "memory_banks"
TABLES_DIR = OUTPUT_DIR / "tables"
FIGURES_DIR = OUTPUT_DIR / "figures"

for d in [METRICS_DIR, MEMORY_DIR, TABLES_DIR, FIGURES_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("Output folders created.")

## 5. Helper Functions

In [ ]:
def build_extractor(extractor_type: str, device: torch.device) -> torch.nn.Module:
    """Instantiate and freeze the feature extractor."""
    if extractor_type == "layer3":
        extractor = FeatureExtractor().to(device)
    elif extractor_type == "multiscale":
        extractor = MultiScaleFeatureExtractor().to(device)
    else:
        raise ValueError("extractor_type must be 'layer3' or 'multiscale'")
    extractor.eval()
    return extractor

In [ ]:
def evaluate_category(
    category: str,
    extractor_type: str,
    device: torch.device,
    data_dir: str,
    batch_size: int = 32,
    use_random_coreset: bool = False,
    coreset_ratio: float = 0.1,
    k_ratio: float = 0.01,
) -> tuple[dict, torch.Tensor]:
    """
    Build memory bank and compute AUROC for one MVTec category.

    Returns:
        category_results (dict): AUROC per scoring method + metadata
        memory_bank (Tensor): the built memory bank (on CPU)
    """
    print(f'\n{'='*50}')
    print(f'  Category: {category}')
    print(f'{'='*50}')

    # ── Datasets ──
    train_dataset = MvtecAdDataset(
        root_dir=data_dir, category=category, split='train', transform=preprocess_image
    )
    test_dataset = MvtecAdDataset(
        root_dir=data_dir, category=category, split='test', transform=preprocess_image
    )
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=False)
    test_loader  = DataLoader(test_dataset,  batch_size=batch_size, shuffle=False)

    print(f'  Train: {len(train_dataset)} images | Test: {len(test_dataset)} images')

    #  Feature extractor 
    extractor = build_extractor(extractor_type, device)

    #  Memory bank
    memory_bank = build_memory_bank(
        feature_extractor=extractor,
        dataloader=train_loader,
        device=device,
    )
    print(f'  Memory bank (full): {tuple(memory_bank.shape)}')

    if use_random_coreset:
        memory_bank = random_coreset_sampling(
            memory_bank=memory_bank, sampling_ratio=coreset_ratio
        )
        print(f'  Memory bank (coreset): {tuple(memory_bank.shape)}')

    # Evaluation 
    category_results = {
        'category':           category,
        'extractor_type':     extractor_type,
        'train_size':         len(train_dataset),
        'test_size':          len(test_dataset),
        'memory_bank_size':   memory_bank.shape[0],  # int — JSON-safe
        'memory_bank_dim':    memory_bank.shape[1],  # int — JSON-safe
    }

    for method in SCORING_METHODS:
        scores, labels = compute_patchcore_scores(
            feature_extractor=extractor,
            dataloader=test_loader,
            memory_bank=memory_bank,
            device=device,
            reduction=method,
            k_ratio=k_ratio,
        )
        auc = roc_auc_score(labels, scores)
        category_results[method] = round(float(auc), 4)
        print(f'  {method:<12}: AUROC = {auc:.4f}')

    return category_results, memory_bank.cpu()

## 6. Benchmark Loop

Results are saved incrementally after each category — safe against Colab session crashes.

In [ ]:
all_results = []

for category in tqdm(CATEGORIES, desc="Benchmark", unit="category"):
    try:
        category_results, memory_bank = evaluate_category(
            category=category,
            extractor_type=EXTRACTOR_TYPE,
            device=DEVICE,
            data_dir=DATA_DIR,
            batch_size=BATCH_SIZE,
            use_random_coreset=USE_RANDOM_CORESET,
            coreset_ratio=CORESET_RATIO,
            k_ratio=K_RATIO,
        )
        all_results.append(category_results)

        # save memory bank
        if SAVE_MEMORY_BANKS:
            mb_path = MEMORY_DIR / f"{RUN_NAME}_{category}_{EXTRACTOR_TYPE}.pt"
            torch.save(
                {
                    "category": category,
                    "extractor_type": EXTRACTOR_TYPE,
                    "memory_bank": memory_bank,
                },
                mb_path,
            )
            print(f"  Memory bank saved: {mb_path}")

            # Incremental save (crash-safe)
            partial_df = pd.DataFrame(all_results)
            partial_csv = TABLES_DIR / f"{RUN_NAME}_partial.csv"
            partial_df.to_csv(partial_csv, index=False)

    except Exception as e:
        print(f"\n  ERROR on {category}: {e}")
        all_results.append({"category": category, "error": str(e)})

print("\n Benchmark complete.")

## 7. Results Table

In [ ]:
results_df = pd.DataFrame(all_results)

# Keep only successful rows for display
auroc_cols = ["mean", "max", "topk_mean"]
valid_df = results_df.dropna(subset=auroc_cols).copy()
valid_df = valid_df.sort_values(by="topk_mean", ascending=False).reset_index(drop=True)

display_cols = ["category", "train_size", "test_size"] + auroc_cols
valid_df[display_cols].style.format(
    {col: "{:.4f}" for col in auroc_cols}
).background_gradient(subset=auroc_cols, cmap="RdYlGn", vmin=0.5, vmax=1.0)

## 8. Summary Statistics

In [ ]:
summary = {
    "n_categories": len(valid_df),
    "mean_auroc_mean": round(valid_df["mean"].mean(), 4),
    "mean_auroc_max": round(valid_df["max"].mean(), 4),
    "mean_auroc_topk": round(valid_df["topk_mean"].mean(), 4),
    "best_category": valid_df.iloc[0]["category"],
    "best_auroc_topk": valid_df.iloc[0]["topk_mean"],
    "worst_category": valid_df.iloc[-1]["category"],
    "worst_auroc_topk": valid_df.iloc[-1]["topk_mean"],
}

print("Summary")
print("-" * 40)
for k, v in summary.items():
    print(f"  {k:<25}: {v}")

## 9. Visualization - AUROC by Category

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))

x = range(len(valid_df))
width = 0.25
colors = ["#4C72B0", "#DD8452", "#55A868"]

for i, (method, color) in enumerate(zip(auroc_cols, colors)):
    offset = (i - 1) * width
    bars = ax.bar(
        [xi + offset for xi in x],
        valid_df[method],
        width=width,
        label=method,
        color=color,
        alpha=0.85,
    )

ax.axhline(y=0.5, color="red", linestyle="--", linewidth=1, label="Random (0.5)")
ax.set_xticks(list(x))
ax.set_xticklabels(valid_df["category"], rotation=45, ha="right", fontsize=10)
ax.set_ylabel("AUROC")
ax.set_ylim(0.0, 1.05)
ax.set_title(f"PatchCore AUROC per Category - {EXTRACTOR_TYPE} extractor", fontsize=13)
ax.legend()
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()

fig_path = FIGURES_DIR / f"{RUN_NAME}_auroc_by_category.png"
plt.savefig(fig_path, dpi=150)
plt.show()
print("Figure saved:", fig_path)

## 10. Save Final Results

In [ ]:
if SAVE_RESULTS:
    # CSV
    csv_path = TABLES_DIR / f"{RUN_NAME}.csv"
    valid_df.to_csv(csv_path, index=False)
    print("CSV saved:", csv_path)

    # JSON - fully serializable
    json_path = METRICS_DIR / f"{RUN_NAME}.json"
    payload = {
        "run_name": RUN_NAME,
        "extractor_type": EXTRACTOR_TYPE,
        "use_random_coreset": USE_RANDOM_CORESET,
        "coreset_ratio": CORESET_RATIO if USE_RANDOM_CORESET else None,
        "k_ratio": K_RATIO,
        "categories": CATEGORIES,
        "summary": summary,
        "results": all_results,
    }
    with open(json_path, "w") as f:
        json.dump(payload, f, indent=4)
    print("JSON saved:", json_path)

    # Remove partial file now that we have the final
    partial = TABLES_DIR / f"{RUN_NAME}_partial.csv"
    if partial.exists():
        partial.unlink()
        print("Partial CSV removed.")

## 11. Analysis - Easy vs Hard Categories

In [ ]:
THRESHOLD_EASY = 0.90
THRESHOLD_HARD = 0.70

easy_categories = valid_df[valid_df["topk_mean"] >= THRESHOLD_EASY]["category"].tolist()
hard_categories = valid_df[valid_df["topk_mean"] < THRESHOLD_HARD]["category"].tolist()
mid_categories = valid_df[
    (valid_df["topk_mean"] >= THRESHOLD_HARD) & (valid_df["topk_mean"] < THRESHOLD_EASY)
]["category"].tolist()

print(f"Easy (AUROC >= {THRESHOLD_EASY}): {easy_categories}")
print(f"Medium ({THRESHOLD_HARD} <= AUROC < {THRESHOLD_EASY}): {mid_categories}")
print(f"Hard  (AUROC < {THRESHOLD_HARD}): {hard_categories}")

## 13. Qualitative Analysis - Anomaly Heatmaps

Visualize PatchCore anomaly maps on representative categories:
- an easy category (bottle)
- a medium category (capsule)
- a hard category (grid)

Each row shows: original image | anomaly heatmap | overlay.

In [ ]:
from visualization.heatmap import visualize_patchcore_overlay

# Categories to visualize — one easy, one medium, one hard
VIZ_CATEGORIES = [
    ("bottle", "Easy   — AUROC 0.999"),
    ("capsule", "Medium — AUROC 0.878"),
    ("grid", "Hard   — AUROC 0.577"),
]

NUM_IMAGES = 5
OVERLAY_ALPHA = 0.45

for category, label in VIZ_CATEGORIES:
    print(f'\n{"="*50}')
    print(f"  {label} — {category}")
    print(f'{"="*50}')

    # Dataset
    test_dataset = MvtecAdDataset(
        root_dir=DATA_DIR,
        category=category,
        split="test",
        transform=preprocess_image,
    )
    test_loader = DataLoader(test_dataset, batch_size=NUM_IMAGES, shuffle=False)

    # Feature extractor + memory bank for this category
    extractor = build_extractor(EXTRACTOR_TYPE, DEVICE)

    train_dataset = MvtecAdDataset(
        root_dir=DATA_DIR,
        category=category,
        split="train",
        transform=preprocess_image,
    )
    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=False)

    memory_bank = build_memory_bank(
        feature_extractor=extractor,
        dataloader=train_loader,
        device=DEVICE,
    )

    if USE_RANDOM_CORESET:
        memory_bank = random_coreset_sampling(
            memory_bank=memory_bank,
            sampling_ratio=CORESET_RATIO,
        )

    # Visualize
    visualize_patchcore_overlay(
        feature_extractor=extractor,
        dataloader=test_loader,
        memory_bank=memory_bank,
        device=DEVICE,
        num_images=NUM_IMAGES,
        reduction="topk_mean",
        k_ratio=K_RATIO,
        alpha=OVERLAY_ALPHA,
    )